In [1]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv()
hf_token = os.environ["HUGGINGFACE_HUB_TOKEN"]
login(token=hf_token)
print(len(hf_token) != 0)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/long-t5-tglobal-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [3]:
import json
with open("transcript.txt", "r", encoding="utf-8") as f:
    transcript = f.read().split("\n")

In [4]:
tokens = [tokenizer(line) for line in transcript]

Semantic chunker libraries tend to just split by newline, so making our own chunker that takes in multiple lines for context, but keeps lines intact.

In [5]:
def token_length(token):
    return len(token.input_ids)

def chunk_text(lines, max_tokens=64, min_overlap=16):
    tokens = [tokenizer(line) for line in lines]
    for token in tokens:
        if token_length(token) > max_tokens:
            raise ValueError("Line has more tokens than max tokens")
    
    chunks = []
    start = next_start = end = curr_length = 0
    
    while end < len(tokens):
        while end < len(tokens) and (curr_length + token_length(tokens[end])) <= max_tokens:
            curr_length = curr_length + token_length(tokens[end])
            end += 1
            if max_tokens - curr_length >= min_overlap:
                next_start = end
        chunks.append([tokenizer.decode(token.input_ids, skip_special_tokens=True) for token in tokens[start:end]])
        start = end = next_start
        curr_length = 0

    return chunks

def chunk_length(chunk):
    return sum(token_length(token) for token in chunk)
    

In [6]:
chunks = chunk_text(transcript)

In [7]:
import torch
def summarize(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=64)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=1024,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            early_stopping=True
        )
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return summary

In [8]:
summarized_chunks = {
    summarize("\n".join(chunk)): chunk
    for chunk in chunks
}

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.10/dist-packages/apex/normalization/fused_layer_norm.py:214: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


In [9]:
import json
with open("chunks.json", "w") as f:
    json.dump(summarized_chunks, f)